In [1]:
import psutil
print("Total RAM:", psutil.virtual_memory().total / (1024 ** 3), "GB")
print("Available RAM:", psutil.virtual_memory().available / (1024 ** 3), "GB")


Total RAM: 15.845481872558594 GB
Available RAM: 9.084457397460938 GB


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3, InceptionResNetV2
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# بارگذاری داده‌های CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# تغییر اندازه تصاویر به 75x75 (حداقل اندازه مورد نیاز برای InceptionV3)
def resize_images(images):
    return tf.image.resize(images, [75, 75])

x_train_resized = resize_images(x_train).numpy()
x_test_resized = resize_images(x_test).numpy()

# نرمال‌سازی داده‌ها
x_train_resized = x_train_resized / 255.0
x_test_resized = x_test_resized / 255.0

# تبدیل برچسب‌ها به one-hot encoding
y_train_categorical = to_categorical(y_train, 10)
y_test_categorical = to_categorical(y_test, 10)

In [4]:
def create_inception_model():
    # بارگذاری مدل پایه با وزن‌های ImageNet (بدون لایه‌های بالایی)
    base_model = InceptionV3(weights='imagenet', 
                            include_top=False, 
                            input_shape=(75, 75, 3))
    
    # انجماد لایه‌های پایه (اختیاری - می‌توانید این بخش را حذف کنید)
    for layer in base_model.layers:
        layer.trainable = True
    
    # ایجاد مدل جدید روی پایه
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    # کامپایل مدل
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

model = create_inception_model()
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 inception_v3 (Functional)   (None, 1, 1, 2048)        21802784  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 10)                2570      
                                                                 
Total params: 22,329,898
Trainable params: 22,295,466
Non-trainable params: 34,432
_______________________________________

In [7]:
# تعریف callback ها
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

# آموزش مدل با batch_size=32
history = model.fit(
    x_train_resized, y_train_categorical,
    batch_size=16,  # کاهش batch size
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [7]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_gpu_available())
print(tf.test.gpu_device_name())

Num GPUs Available:  1
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
True
/device:GPU:0


In [6]:
train_dataset = tf.data.Dataset.from_tensor_slices((x_train_resized, y_train_categorical))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(32)

history = model.fit(
    train_dataset,
    epochs=50,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [9]:
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))


2.10.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [1]:
# @title String fields

text = 'value' # @param {type:"string"}
dropdown = '1st option' # @param ["1st option", "2nd option", "3rd option"]
text_and_dropdown = 'value' # @param ["1st option", "2nd option", "3rd option"] {allow-input: true}
text_with_placeholder = '' # @param {type:"string", placeholder:"enter a value"}

print(text)
print(dropdown)
print(text_and_dropdown)
print(text_with_placeholder)

value
1st option
value

